## **Extracción de Características Visuales (Visual Embeddings)**

En este notebook abordamos la fase de **extracción de características** para la modalidad de vídeo. El objetivo principal es transformar la información "cruda" de los vídeos (píxeles) en representaciones compactas de significado (**embeddings**), que posteriormente alimentarán a los modelos de Deep Learning en la fase de experimentación.

Este paso es clave en el pipeline ya que los vídeos originales de ambos *datasets* contienen una gran cantidad de información redundante (miles de píxeles por frame, repetidos 30 veces por segundo, por ejemplo). Entrenar nuestros modelos directamente con los píxeles en crudo es computacionalmente inviable y puede derivar al *overfitting* de nuestros modelos. Por tanto, aplicamos una estrategia de **transfer learning** donde utilizaremos redes neuronales convolucionales (CNNs) y transformers ya pre-entrenadas en millones de imágenes (ImageNet) para que actúen como "extractores de características". Estas redes ya *saben ver* (detectan bordes, formas, texturas y patrones faciales complejos), por lo que podemos usar sus capas internas para resumir cada frame del vídeo en un vector numérico de alta densidad.

Para ello, siguiendo la metodología, extraemos las características de **tres arquitecturas diferentes seleccionadas** para poder comparar su rendimiento:

1. **ResNet50 (Residual Networks):** Estándar en la industria de la Visión Artificial. Utiliza conexiones residuales para permitir redes muy profundas. Genera vectores de **2048 dimensiones** por defecto.

2. **EfficientNet-B0:** Arquitectura diseñada para ser extremadamente eficiente en recursos computacionales sin perder precisión. Genera vectores de **1280 dimensiones** por defecto, lo que ahorrará espacio y tiempo de cómputo.

3. **ViT-B/16 (Vision Transformer):** Arquitectura base basada en Transformer que ha demostrado excelente rendimiento en tareas de visión artificial. Procesa las imágenes como secuencias de parches y genera vectores de **768 dimensiones** por defecto. Se ha seleccionado la versión **Base** de **ViT** en lugar del **Large** (1024) ya que, como ya hemos justificado, los vídeos de MELD e IEMOCAP son vídeos cortos en su mayoría, por tanto una dimensión de 1024 introduciría demasiada redundancia, lo que provocaría que la red hiciera *overfitting*.


**NOTA:** Se tienen en cuenta los resultados obtenidos en el EDA del número de *frames* para una muestra aleatoria tanto en MELD como en IEMOCAP, para realizar un **análisis previo** y así decidir la resolución de muestreo. 

In [7]:
# Carga previa de todas las librerías y paquetes necesarios
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn as nn # Para modificar las capas finales de los modelos (eliminarlas) 
from PIL import Image
from tqdm import tqdm # Para barra de progreso
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader

# Configuración de GPU/CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

Dispositivo: cuda


Definimos las rutas desde las cuales se cargan los vídeos en el servidor DGX:

In [8]:
BASE_DIR = os.path.expanduser('/workspace') #Directorio base
CSV_PATH = os.path.join(BASE_DIR, 'Multimodal_Stress_Dataset.csv') # CSV global unificado
LOCAL_DATA_ROOT = os.path.join(BASE_DIR, 'EMBEDDINGS_VISUAL') # Directorio donde se almacenarán los features visuales
os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

# Carga del dataset global
if os.path.exists(CSV_PATH):
    df_global = pd.read_csv(CSV_PATH)
    display(df_global.head()) 
else:
    print(f"No se encuentra el archivo en {CSV_PATH}")

,Utterance_ID,Dialogue_ID,video_path,audio_path,Transcription,duration,split,target_stress,dataset_origin
0,train_dia0_utt0,0,train_splits/dia0_utt0.mp4,MELD_Audio/train_dia0_utt0.wav,also I was the point person on my company's tr...,5.672333,train,0,MELD
1,train_dia0_utt1,0,train_splits/dia0_utt1.mp4,MELD_Audio/train_dia0_utt1.wav,You must've had your hands full.,1.501500,train,0,MELD
2,train_dia0_utt2,0,train_splits/dia0_utt2.mp4,MELD_Audio/train_dia0_utt2.wav,That I did. That I did.,2.919583,train,0,MELD
3,train_dia0_utt3,0,train_splits/dia0_utt3.mp4,MELD_Audio/train_dia0_utt3.wav,So let's talk a little bit about your duties.,2.752750,train,0,MELD
4,train_dia0_utt4,0,train_splits/dia0_utt4.mp4,MELD_Audio/train_dia0_utt4.wav,My duties? All right.,6.464792,train,0,MELD


-----

#### **ESTRUCTURA NOTEBOOK Y EJECUCIÓN**:

Debido a la alta carga computacional que supone el procesado de los *clips* de vídeo de ambos datasets para la extracción de los embeddings, y a esto se le suman los **3 diferentes codificadores** que utilizaremos para ello, se ha optado por realizar la carga y ejecución en el Servidor NVIDIA DGX de **CyberDataLab**, al cual se ha tenido acceso para realizar el TFG. 

A continuación, se presenta el análisis inicial llevado a cabo, previo a la extracción de características visuales, y se explica y ejecuta el código de inicialización de cada modelo a utilizar (**ResNet50**, **EfficientNet-B0** y **ViT-B/16**), así como el bucle final de ejecución lanzado en el servidor para la extracción de características. 

Finalmente, se cargan los *embeddings* resultantes para realizar una auditoría final de validación (**sanity check**).


---

### **Análisis Previo**

* Identificamos el número de *frames* promedio en una muestra aleatoria para justificar la decisión de cuántos *frames* se extraerán del vídeo:

Se vuelven a indicar los resultados obtenidos en el EDA para MELD e IEMOCAP:

Resultados Frames en MELD:\
Media de frames por vídeo: 87.42\
Mínimo: 2.00, Máximo: 231.00

Resultados Frames en IEMOCAP:\
Media de frames por vídeo: 123.14\
Mínimo: 34.00, Máximo: 382.00

Para dedicir el tamaño de la ventana (número de frames), se tiene también en cuenta el análisis realizado por Paul Ekman en cuanto a la duración de las microexpresiones faciales (presente en su libro *Emotions revealed: recognizing faces and feelings to improve communication and emotional life*, URL: https://books.google.es/books?id=JJ9xHXo7hPgC&redir_esc=y). Este demuestra que las microexpresiones ocurren automáticamente en los seres humanos y delatan la emoción real que está sintiendo. Estas ocurren entre 1/25 o 1/5 de segundo, es decir, entre 40ms - 200ms, no llegando a alcanzar la mayoría de ella los 200ms. Por tanto, en cuanto a este razonamiento, nuestro objetivo es capturar al menos un frame en cada intervalo menor a 200 ms, capturando así mucho mejor los instantes representativos dentro del intervalo de la microexpresión, y eliminando frames redundantes. Ekman, en su libro, descarta que el estrés sea una simple emoción (él define las emociones como aquellas que sí cuentan con microexpresiones claras), sino que habla del estrés como un estado sostenido con múltiples señales, siendo las microexpreciones indicadores puntuales de la misma. Por tanto, al tener una mayor duración que simplemente una emoción, con capturar al menos un frame en dicho intervalo de la microexpresión es suficiente. 

Para lograrlo, tenemos en cuenta los resultados obtenidos del EDA: 
- La duración media de los clips de vídeo en **MELD** es de aproximadamente 3s (con un número medio de frames de 87.42).

- La duración media de los clips de víedo en **IEMOCAP** es de aproximadamente 5s (con un número medio de frames de 123.14).

Tomando como referencia la duración mayor (5s en IEMOCAP), si fijamos la captura de 32 frames por vídeo, cumplimos con el intervalo indicado por Ekman (5000ms / 32 frames = 156.25 ms < 200 ms). Además, con 32 frames, se permite capturar en **IEMOCAP** aproximadamente el **26%** de la información y un **36.6%** en **MELD** sin tener que explotar la **RAM**. Esta configuración equivale a una frecuencia de muestreo de aproximadamente 6-8 FPS, lo cual proporciona un equilibrio adecuado entre resolución y coste computacional, permitiendo capturar esos patrones más sostenidos asociados al estrés.

**CONCLUSIÓN**: Con **32 frames** (`num_frames = 32`), el tensor resultante tendrá dimensiones `(32,dim_model)` (dependiendo del modelo), garantizando un mayor balance entre ganularidad y viabilidad de memoria para el entrenamiento de las redes recurrentes (**LSTM**). 

**NOTA**: Para el ajuste de hiperparámetros, se probará con un valor menor a 32 (**16**) para comparar el rendimiento que obtiene el modelo con una frecuencia de muestreo menor.

----

La estructura de directorios, tras la ejecución de este notebook, quedará de la siguiente forma:
* **`EMBEDDINGS_VISUAL`**:

    * **`RESNET`**: 

        * **`MELD`**: Embeddings de tamaño $(32,2048)$ obtenidos de **ResNet50** para el dataset de **MELD**.

        * **`IEMOCAP`**: Embeddings de tamaño $(32,2048)$ obtenidos de **ResNet50** para el dataset de **IEMOCAP**.


    * **`EFFICIENTNET`**: 

        * **`MELD`**: Embeddings de tamaño $(32,1280)$ obtenidos de **EfficientNet-B0** para el dataset de **MELD**.

        * **`IEMOCAP`**: Embeddings de tamaño $(32,1280)$ obtenidos de **EfficientNet-B0** para el dataset de **IEMOCAP**.

    
    * **`VIT`**: 

        * **`MELD`**: Embeddings de tamaño $(32,768)$ obtenidos de **ViT-B/16** para el dataset de **MELD**.

        * **`IEMOCAP`**: Embeddings de tamaño $(32,768)$ obtenidos de **ViT-B/16** para el dataset de **IEMOCAP**.

---
## **ResNet50**

In [9]:
def resnet_extractor():
    """
    Carga ResNet50 pre-entrenada y elimina la capa de clasificación (FC).
    Devuelve:
    - model: El modelo extractor listo para usar.
    - output_dim: 2048 (Número de características por frame)
    """
    
    # Cargamos los pesos entrenados en ImageNet. Esto asegura que la red sepa ya reconocer formas, luces y texturas (sesgo inductivo de la CNN):
    weights = models.ResNet50_Weights.IMAGENET1K_V1 ### ResNet50_Weights.DEFAULT == ResNet50_Weights.IMAGENET1K_V2, por eso no hemos elegido la DEFAULT, ya que la V2 es una versión mejorada del pipeline de entrenamiento (sobre el mismo dataset de ImageNet-1K), lo que haría la comparativa con respecto a ViT y a EfficientNet injusta. Con V1 seleccionamos los pesos obtenidos del modelo con el pipeline estándar básico de entrenamiento (sobre el mismo dataset ImageNet-1K) 
    model = models.resnet50(weights=weights)
    
    # ----- ELIMINAMOS ÚLTIMA CAPA DE CLASIFICACIÓN -----
    # La arquitectura ResNet original termina en una capa 'fc' (Linear) que clasifica en 1000 clases.
    # Nosotros queremos el vector justo antes de esa clasificación
    modules = list(model.children())[:-1] # Con list(model.children())[:-1] cogemos todas las capas menos la última
    model = nn.Sequential(*modules) # Creamos un nuevo modelo con todas las capas excepto la última
    
    # Congelamos los pesos (Freeze), ya que no queremos entrenar la CNN, solo usarla directamente:
    for param in model.parameters():
        param.requires_grad = False
        
    model.to(device)
    model.eval() # Ponemos en modo evaluación 
    
    return model, 2048  # ResNet50 siempre devuelve vectores de tamaño 2048

# Definimos también las transformaciones que ResNet (al igual que EfficientNet y ViT) necesita para funcionar:
# Son reglas estrictas de ImageNet: normalización con media/std específicos
preprocess = transforms.Compose([ 
    transforms.Resize(256), # Redimensionamos a 256x256
    transforms.CenterCrop(224), # Recortamos al centro para quedarnos con 224x224, que es el tamaño que ResNet y EfficientNet esperan
    transforms.ToTensor(), # Convertimos a tensor (esto también escala los píxeles a [0,1])
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Normalización específica de ImageNet
])

# Con esto, ya tenemos lista la configuración y preprocesamiento de los frames para ResNet

-------
## **EfficientNet-B0**

In [10]:
def efficientnet_extractor():
    """
    Carga EfficientNet-B0 y adapta para extracción de características.
    Devuelve:
    - model: El extractor.
    - output_dim: 1280
    """
    
    #  Cargamos los pesos
    weights = models.EfficientNet_B0_Weights.DEFAULT ## EfficientNet_B0_Weights.DEFAULT == EfficientNet_B0_Weights.IMAGENET1K_V1, en este caso no tenemos la V2 disponible, pero obtenemos un modelo básico también entrenado sobre el mismo dataset ImageNet-1K
    model = models.efficientnet_b0(weights=weights)
    
    # -----ELIMINAMOS ÚLTIMA CAPA DE CLASIFICACIÓN -----
    # En EfficientNet, la capa de clasificación se llama 'classifier'
    # Únicamente queremos lo que se obtiene de las 'features' y pasa por el 'avgpool':
    model = nn.Sequential(
        model.features,  #Selecionamos las features 
        model.avgpool  # y avgpool, par obtener finalmente un vector 1D
    )
    
    # Congelamos los pesos:
    for param in model.parameters():
        param.requires_grad = False
        
    model.to(device)
    model.eval()
    
    return model, 1280  # EfficientNet-B0 devuelve vectores de 1280

# Se definen las mismas transformaciones que para ResNet (preprocess), ya que es el estándar en ImageNet
# Con esto ya tenemos configurado EfficientNet-B0

-------
## **ViT (Vision Transformer)**

In [11]:
def vit_extractor():
    """
    Carga ViT-B/16 pre-entrenado y adapta para extracción de características.
    Devuelve:
    - model: El extractor.
    - output_dim: 768
    """
    
    # Cargamos los pesos pre-entrenados de ViT-B/16
    weights = models.ViT_B_16_Weights.DEFAULT ## ViT_B_16_Weights.DEFAULT == ViT_B_16_Weights.IMAGENET1K_V1, obtenemos el modelo básico entrenado sobre el mismo dataset de ImageNet-1K (no está disponible la V2)
    model = models.vit_b_16(weights=weights)
    # -----ELIMINAMOS ÚLTIMA CAPA DE CLASIFICACIÓN -----
    # En ViT, la capa de clasificación se llama 'head'. Para extraer las características directamente, 
    # podemos reemplazar esta capa por una identidad, lo que hará que el modelo devuelva directamente el vector de características (el CLS token)
    model.heads = nn.Identity()
    
    # Congelamos los pesos:
    for param in model.parameters():
        param.requires_grad = False
        
    model.to(device)
    model.eval()
    
    return model, 768  # ViT-B/16 devuelve vectores de 768

# Las transformaciones son las mismas que para ResNet y EfficientNet
# Con esto ya tenemos configurado ViT-B/16

----
### ***Feed-Forward*. Extracción de características.**

Una vez ya tenemos definido y cargados los modelos, extraemos los embeddings de cada uno de los vídeos (y los almacenamos en los directorios creados):

In [13]:
# ------------------------------------------------------------------------
# 1. Definición del Dataset para Carga Paralela (Multiprocessing)
# ------------------------------------------------------------------------
class VideoDataset(Dataset):
    def __init__(self, df, root_meld, root_iemocap, transform=None):
        self.df = df
        self.root_meld = root_meld
        self.root_iemocap = root_iemocap
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Selección de ruta según el origen del vídeo
        if row['dataset_origin'] == 'MELD':
            video_path = os.path.join(self.root_meld, row['video_path'])
        else:
            video_path = os.path.join(self.root_iemocap, row['video_path'])
        
        # Extracción de 32 frames uniformes del vídeo
        frames = self.extract_frames(video_path)
        if self.transform and len(frames) > 0:
            frames = torch.stack([self.transform(f) for f in frames]) # torch.stack --> (num_frames, canales, alto, ancho)

        
        # Retornamos el tensor de frames y el nombre formateado para el archivo .npy
        file_name = f"{row['dataset_origin']}_{row['Utterance_ID']}.npy".replace("/", "_")
        return frames, file_name

    def extract_frames(self, video_path, num_frames=32):
        """
        Abre un vídeo y extrae 'num_frames' imágenes distribuidas uniformemente.
        Devuelve: Una lista de imágenes en formato PIL.
        """
        # Abrimos el vídeo con OpenCV (en formato BGR, que es el formato nativo de OpenCV y evita gasto de CPU en convertir a RGB si no es necesario)
        cap = cv2.VideoCapture(video_path)
    
        # Verificamos si el vídeo abre
        if not cap.isOpened():
            return [] # Si no se puede abrir el vídeo, devolvemos una lista vacía

        # Leemos todos los frames del vídeo y los almacenamos en una lista:
        all_frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break # Si no se pueden leer más frames, salimos del bucle, ya que hemos leído todos los frames disponibles
            all_frames.append(frame) # Guardamos BGR 
    
        cap.release() # Cerramos el vídeo para liberar recursos

        total_frames = len(all_frames)
        if total_frames == 0:
            return []  # Por seguridad, si el vídeo no tiene frames, devolvemos una lista vacía

        # Calculamos qué índices coger, con np.linspace creamos una secuencia espaciada uniformemente (ej: 0, 5, 10...)
        indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
        # Esto asegura que aunque el vídeo tenga menos de 'num_frames', no se generarán índices fuera del rango, y se repetirá el último frame según sea necesario
    
        # Solo convertimos a RGB y PIL los 32 frames seleccionados, para evitar gasto de CPU en convertir a RGB los frames que no vamos a usar:
        selected_frames = []
        for i in indices:
            frame_bgr = all_frames[i]
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            selected_frames.append(Image.fromarray(frame_rgb))
    
        # Padding de seguridad si el video era muy corto (menos de 32 frames), repetimos el último frame hasta completar los 32:
        while len(selected_frames) < num_frames:
            selected_frames.append(selected_frames[-1])
        
        return selected_frames

# -------------------------------------------------------------------------
# 2. Lógica Principal de Extracción
# -------------------------------------------------------------------------
def main_extraction(df_subset, origin, output_dir, model_name, preproc, root_meld_path, root_iemocap_path, num_workers=8):
    """
    Función principal de extracción diseñada para ejecución paralela en Servidor DGX.
    Args:
    - df_subset: DataFrame filtrado por dataset individual seleccionado (MELD o IEMOCAP).
    - origin: nombre dataset individual seleccionado (MELD o IEMOCAP)
    - output_dir: Nombre del directorio de salida donde se almacenarán los embeddings
    - model_name (str): Nombre del modelo a usar ('resnet', 'efficientnet' o 'vit').
    - preproc: pipeline de preprocesamiento para el modelo
    - root_meld_path: ruta del directorio donde se encuentran los clips de vídeo para MELD
    - root_iemocap_pah: ruta del directorio donde se encuentran los clips de vídeo para IEMOCAP
    - num_workers (int): Número de procesos paralelos para DataLoader.
    Devuelve:
    - Guarda los embeddings extraídos en archivos .npy en el directorio correspondiente.
    """
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(output_dir, exist_ok=True)

    # 1. Configuración del Extractor según Arquitectura seleccionada:
    if model_name == 'resnet':
        model, dim = resnet_extractor()
    elif model_name == 'efficientnet':
        model, dim = efficientnet_extractor()
    elif model_name == 'vit':
        model, dim = vit_extractor()

    # 2. Carga de datos (solo del subset):
    dataset = VideoDataset(df_subset, root_meld_path, root_iemocap_path, transform=preproc)
    dataloader = DataLoader(dataset, batch_size=1, num_workers=num_workers)

    # 3. Extracción paralela:
    with torch.no_grad():
        for frames, name in tqdm(dataloader, desc=f"Extrayendo {model_name} - {origin}"):
            if frames.numel() == 0:  #con .numel devuleve el número de elementos del tensor, si este está vacío, devuelve 0
                continue

            if os.path.exists(os.path.join(output_dir, name[0])): # Si el embedding ya estaba creado, lo salta
                continue
            
            frames = frames.squeeze(0).to(device) # Shape: (32, 3, 224, 224)
            embeddings = model(frames)
            # Embeddings shape tras pasar por los modelos:
            # ResNet/EfficientNet -> (32, Dim, 1, 1) -> Necesitamos aplicar flatten
            # ViT (con Wrapper) -> (32, 768)  -> Ya está plano
            
            if model_name != 'vit':
                embeddings = embeddings.flatten(start_dim=1)
            
            np.save(os.path.join(output_dir, name[0]), embeddings.cpu().numpy())
    




# -------------------------------------------------------------------------
# EJECUCIÓN
# -------------------------------------------------------------------------

if __name__ == '__main__':
    modelos = ['resnet', 'efficientnet', 'vit']
    origenes = ['MELD', 'IEMOCAP']

    for modelo in modelos:
        print(f"\n---------Extracción con modelo: {modelo.upper()}----------\n")
        for origen in origenes:
            df_filtrado = df_global[df_global['dataset_origin']==origen].copy()

            output_dir_path = os.path.join(LOCAL_DATA_ROOT, modelo.upper(), origen.upper())
        
            main_extraction(
                df_subset=df_filtrado,
                origin=origen,
                output_dir = output_dir_path,
                model_name = modelo,
                preproc = preprocess,
                root_meld_path = "./data/MELD_CLIPS",
                root_iemocap_path = "./data"
            )



---------Extracción con modelo: RESNET----------



Extrayendo resnet - IEMOCAP: 100%|██████████████████████████████████████████████████| 7515/7515 [06:08<00:00, 20.38it/s]



---------Extracción con modelo: EFFICIENTNET----------

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████████████████████████████████████████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 71.2MB/s]
Extrayendo efficientnet - IEMOCAP: 100%|████████████████████████████████████████████| 7515/7515 [05:09<00:00, 24.29it/s]



---------Extracción con modelo: VIT----------

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|█████████████████████████████████████████████████████████████████████████████████| 330M/330M [00:03<00:00, 112MB/s]
Extrayendo vit - IEMOCAP: 100%|█████████████████████████████████████████████████████| 7515/7515 [08:33<00:00, 14.64it/s]


### **Sanity Check**. 

Realizamos a continuación una verficación rápida de que todos los *embeddings* se han obtenido correctamente (en total y por cada corpus/partición).

Este *Sanity Check* realiza lo siguiente:
* Comprueba el número de archivos guardados (debe ser igual a los vídeos indicados en el CSV).
* Muestra un ejemplo de archivo guardado por cada modelo (**ResNet**, **EfficientNet** y **ViT**) para verificar que se han guardado correctamente.
* Se muestran mensajes de error claros en caso de un guardado incorrecto, indicando el número de archivos **esperados** en relación a los **encontrados** (obtenidos).
* Muestra el número de embeddings obtenidos por cada *dataset* (**MELD** e **IEMOCAP**), además de por cada *split*/partición (`train`, `test`, `dev`) para verificar que se han guardado correctamente los archivos de *embeddings* de ambos *datasets* y de cada *split*.

In [14]:
# ----------- SANITY CHECK: VALIDACIÓN DE EMBEDDINGS GUARDADOS ---------

# 1. RECOLECCIÓN DE ARCHIVOS GENERADOS 
print("\n--- 1. VALIDACIÓN DE EMBEDDINGS GUARDADOS ---")
modelos = ['resnet', 'efficientnet', 'vit']
origenes = ['MELD', 'IEMOCAP']

# Usamos set() para que las búsquedas luego sean instantáneas:
saved_files = {'resnet': set(), 'efficientnet': set(), 'vit': set()}
example_paths = {'resnet': None, 'efficientnet': None, 'vit': None}

for modelo in modelos:
    for origen in origenes:
        folder_path = os.path.join(LOCAL_DATA_ROOT, modelo.upper(), origen.upper())
        if os.path.exists(folder_path):
            files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
            saved_files[modelo].update(files) # Añadimos al conjunto global del modelo
            
            # Guardamos una ruta de ejemplo real para el paso 4
            if len(files) > 0 and example_paths[modelo] is None:
                example_paths[modelo] = os.path.join(folder_path, files[0])

total_videos_expected = len(df_global)
print(f"Vídeos esperados (CSV): {total_videos_expected}")
print(f"Embeddings ResNet guardados: {len(saved_files['resnet'])}")
print(f"Embeddings EfficientNet guardados: {len(saved_files['efficientnet'])}")
print(f"Embeddings ViT guardados: {len(saved_files['vit'])}")


# 2. VERIFICACIÓN POR DATASET (MELD vs IEMOCAP)
print("\n--- 2. VALIDACIÓN POR DATASET ---")
for dataset in ['MELD', 'IEMOCAP']:
    dataset_videos = df_global[df_global['dataset_origin'] == dataset]
    expected_count = len(dataset_videos)

    dataset_resnet_files, dataset_efficientnet_files, dataset_vit_files = 0, 0, 0

    for _, row in dataset_videos.iterrows():
        filename = f"{row['dataset_origin']}_{row['Utterance_ID']}.npy".replace("/","_")

        if filename in saved_files['resnet']: 
            dataset_resnet_files += 1
        if filename in saved_files['efficientnet']: 
            dataset_efficientnet_files += 1
        if filename in saved_files['vit']: 
            dataset_vit_files += 1

    print(f"\n{dataset}:")
    print(f"Vídeos esperados: {expected_count}")
    print(f"ResNet embeddings: {dataset_resnet_files} ({dataset_resnet_files/expected_count*100:.1f}%)")
    print(f"EfficientNet embeddings: {dataset_efficientnet_files} ({dataset_efficientnet_files/expected_count*100:.1f}%)")
    print(f"ViT embeddings: {dataset_vit_files} ({dataset_vit_files/expected_count*100:.1f}%)")

    try:
        assert dataset_resnet_files == expected_count, f"Faltan en ResNet"
        assert dataset_efficientnet_files == expected_count, f"Faltan en EfficientNet"
        assert dataset_vit_files == expected_count, f"Faltan en ViT"
    except AssertionError as e:
        print(f"AVISO {dataset}: {str(e)}")


# 3. VERIFICACIÓN POR SPLIT/PARTICIÓN
print("\n--- 3. VALIDACIÓN POR SPLIT/PARTICIÓN ---")
for split in ['train', 'dev', 'test']:
    split_videos = df_global[df_global['split'] == split]
    expected_count = len(split_videos)

    if expected_count == 0: continue

    split_resnet_files, split_efficientnet_files, split_vit_files = 0, 0, 0

    for _, row in split_videos.iterrows():
        filename = f"{row['dataset_origin']}_{row['Utterance_ID']}.npy".replace("/","_")

        if filename in saved_files['resnet']: 
            split_resnet_files += 1
        if filename in saved_files['efficientnet']: 
            split_efficientnet_files += 1
        if filename in saved_files['vit']: 
            split_vit_files += 1

    print(f"\n{split.upper()}:")
    print(f"Vídeos esperados: {expected_count}")
    print(f"ResNet embeddings: {split_resnet_files} ({split_resnet_files/expected_count*100:.1f}%)")
    print(f"EfficientNet embeddings: {split_efficientnet_files} ({split_efficientnet_files/expected_count*100:.1f}%)")
    print(f"ViT embeddings: {split_vit_files} ({split_vit_files/expected_count*100:.1f}%)")


# 4. MOSTRAMOS EJEMPLOS DE EMBEDDINGS GUARDADOS
print("\n--- 4. EJEMPLOS DE EMBEDDINGS GUARDADOS ---")
for modelo in modelos:
    if example_paths[modelo] is not None:
        data = np.load(example_paths[modelo])
        print(f"\n{modelo.upper()} ejemplo: {os.path.basename(example_paths[modelo])}")
        print(f"Shape: {data.shape}")
        print(f"Tamaño: {data.nbytes/1024:.1f} KB")
        print(f"Tipo: {data.dtype}")
   
  


--- 1. VALIDACIÓN DE EMBEDDINGS GUARDADOS ---
Vídeos esperados (CSV): 21219
Embeddings ResNet guardados: 21219
Embeddings EfficientNet guardados: 21219
Embeddings ViT guardados: 21219

--- 2. VALIDACIÓN POR DATASET ---

MELD:
Vídeos esperados: 13704
ResNet embeddings: 13704 (100.0%)
EfficientNet embeddings: 13704 (100.0%)
ViT embeddings: 13704 (100.0%)

IEMOCAP:
Vídeos esperados: 7515
ResNet embeddings: 7515 (100.0%)
EfficientNet embeddings: 7515 (100.0%)
ViT embeddings: 7515 (100.0%)

--- 3. VALIDACIÓN POR SPLIT/PARTICIÓN ---

TRAIN:
Vídeos esperados: 9988
ResNet embeddings: 9988 (100.0%)
EfficientNet embeddings: 9988 (100.0%)
ViT embeddings: 9988 (100.0%)

DEV:
Vídeos esperados: 1108
ResNet embeddings: 1108 (100.0%)
EfficientNet embeddings: 1108 (100.0%)
ViT embeddings: 1108 (100.0%)

TEST:
Vídeos esperados: 2608
ResNet embeddings: 2608 (100.0%)
EfficientNet embeddings: 2608 (100.0%)
ViT embeddings: 2608 (100.0%)

--- 4. EJEMPLOS DE EMBEDDINGS GUARDADOS ---

RESNET ejemplo: MELD_tra

Se confirma que se han generado correctamente todos los *embeddings* **visuales** (este paso es imprescindible para construir la base de nuestros modelos posteriores, ya que estas serán las entradas visuales que recibirán inicialmente). Por tanto, contamos con **21219** embeddings visuales obtenidos con **ResNet50** de tamaño `(32,2048)`, otros **21219** obtenidos con **EfficientNet** de tamaño `(32,1280)`, y **21219** obtenidos con **ViT** de tamaño `(32,768)`.